# Amazon ML Challenge 2026 — FINAL CatBoost Inference

**Purpose:** local, submission-oriented inference using the already-trained **CatBoost V4 Depth-10 production bundle**.

This notebook reuses the baseline inference architecture, but replaces the V1 model and 5-feature recipe with the exact deployed V4 recipe:

- 18 features
- 4 fitted TF-IDF vectorizers
- CatBoost Depth 10
- frozen inference threshold `0.85`

### Important

This notebook **does not**:
- retrain CatBoost
- regenerate blocking
- regenerate the 64 candidate partitions
- rebuild S2/S3 candidate tables

It starts from the existing:

```text
artifacts/final_candidate_partitions/part_000.parquet
...
part_063.parquet
```

and produces:

```text
output/matching_results.tsv
```

The complete Source 1 population is used for the final LEFT JOIN, so every S1 appears exactly once.

In [1]:
# ============================================================
# 1. ENVIRONMENT + PATHS
# ============================================================

from pathlib import Path
import gc
import json
import re
import time

import joblib
import numpy as np
import polars as pl
from rapidfuzz.fuzz import ratio

print("Python environment ready.")

DATA_DIR = Path("./dataset")
ARTIFACT_DIR = Path("./artifacts")
OUTPUT_DIR = Path("./output")
WORK_DIR = Path("./work")

FINAL_CANDIDATE_DIR = ARTIFACT_DIR / "final_candidate_partitions"
INFERENCE_DIR = ARTIFACT_DIR / "inference_matches"

BUNDLE_PATH = ARTIFACT_DIR / "final_tuned_catboost_v4_depth10_deployment_bundle.joblib"

for p in [OUTPUT_DIR, WORK_DIR, INFERENCE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

FILES = {
    "test1": DATA_DIR / "test_source1.tsv",
    "test2": DATA_DIR / "test_source2.tsv",
    "test3": DATA_DIR / "test_source3.tsv",
}

N_PARTITIONS = 64
BATCH_SIZE = 100_000

print("DATA_DIR:", DATA_DIR.resolve())
print("ARTIFACT_DIR:", ARTIFACT_DIR.resolve())
print("FINAL_CANDIDATE_DIR:", FINAL_CANDIDATE_DIR.resolve())
print("BUNDLE_PATH:", BUNDLE_PATH.resolve())

Python environment ready.
DATA_DIR: F:\coding\amazonML2026\submissions\dataset
ARTIFACT_DIR: F:\coding\amazonML2026\submissions\artifacts
FINAL_CANDIDATE_DIR: F:\coding\amazonML2026\submissions\artifacts\final_candidate_partitions
BUNDLE_PATH: F:\coding\amazonML2026\submissions\artifacts\final_tuned_catboost_v4_depth10_deployment_bundle.joblib


In [2]:
# ============================================================
# 2. HARD INPUT / ARTIFACT GATES
# ============================================================

for name, path in FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required dataset file: {path}")
    print(f"✓ {name:6s}: {path}")

if not BUNDLE_PATH.exists():
    raise FileNotFoundError(f"Missing CatBoost deployment bundle: {BUNDLE_PATH}")

candidate_parts = sorted(FINAL_CANDIDATE_DIR.glob("part_*.parquet"))

if len(candidate_parts) != N_PARTITIONS:
    raise RuntimeError(
        f"Expected {N_PARTITIONS} final candidate partitions, "
        f"found {len(candidate_parts)} in {FINAL_CANDIDATE_DIR}"
    )

print(f"✓ Deployment bundle exists: {BUNDLE_PATH.stat().st_size / 1024**2:.2f} MB")
print(f"✓ Final candidate partitions: {len(candidate_parts)}")
print("First:", candidate_parts[0])
print("Last :", candidate_parts[-1])

✓ test1 : dataset\test_source1.tsv
✓ test2 : dataset\test_source2.tsv
✓ test3 : dataset\test_source3.tsv
✓ Deployment bundle exists: 6.91 MB
✓ Final candidate partitions: 64
First: artifacts\final_candidate_partitions\part_000.parquet
Last : artifacts\final_candidate_partitions\part_063.parquet


## 3. Load the production deployment bundle

The Kaggle deployment notebook explicitly packaged:

```text
model
name_vectorizer
address_vectorizer
char_name_vectorizer
char_address_vectorizer
features
threshold
config
```

We will **use those fitted vectorizers directly**. We do not refit TF-IDF locally.

In [3]:
# ============================================================
# 3. LOAD + VERIFY PRODUCTION DEPLOYMENT BUNDLE
# ============================================================

deployment_bundle = joblib.load(BUNDLE_PATH)

required_bundle_keys = [
    "model",
    "name_vectorizer",
    "address_vectorizer",
    "char_name_vectorizer",
    "char_address_vectorizer",
    "features",
    "threshold",
    "config",
]

missing = [key for key in required_bundle_keys if key not in deployment_bundle]
if missing:
    raise RuntimeError(f"Deployment bundle is missing required keys: {missing}")

model = deployment_bundle["model"]
name_vectorizer = deployment_bundle["name_vectorizer"]
address_vectorizer = deployment_bundle["address_vectorizer"]
char_name_vectorizer = deployment_bundle["char_name_vectorizer"]
char_address_vectorizer = deployment_bundle["char_address_vectorizer"]
FEATURES = list(deployment_bundle["features"])
THRESHOLD = float(deployment_bundle["threshold"])

# IMPORTANT:
# The actual production bundle uses the names
#   name_word_tfidf_cosine
#   address_word_tfidf_cosine
# for the two word-TF-IDF features.
# An earlier V4 notebook called the same positional features
#   name_tfidf_cosine / address_tfidf_cosine.
# The numerical features and their order are identical.  The
# deployment bundle itself is the source of truth for feature names.

V4_BASE_FEATURES = [
    "name_edit_similarity",
    "name_token_jaccard",
    "address_edit_similarity",
    "address_token_jaccard",
    "country_match",
    "name_exact_normalized",
    "name_exact_legal",
    "name_length_ratio",
    "address_exact_normalized",
    "address_length_ratio",
    "s1_name_missing",
    "match_name_missing",
    "s1_address_missing",
    "match_address_missing",
]

TFIDF_FEATURE_NAMES_BUNDLE = [
    "name_word_tfidf_cosine",
    "address_word_tfidf_cosine",
    "name_char_tfidf_cosine",
    "address_char_tfidf_cosine",
]

TFIDF_FEATURE_NAMES_LEGACY = [
    "name_tfidf_cosine",
    "address_tfidf_cosine",
    "name_char_tfidf_cosine",
    "address_char_tfidf_cosine",
]

ALLOWED_V4_FEATURE_ORDERS = [
    V4_BASE_FEATURES + TFIDF_FEATURE_NAMES_BUNDLE,
    V4_BASE_FEATURES + TFIDF_FEATURE_NAMES_LEGACY,
]

if FEATURES not in ALLOWED_V4_FEATURE_ORDERS:
    print("Bundle features:", FEATURES)
    print("Expected base features:", V4_BASE_FEATURES)
    raise RuntimeError(
        "Unsupported V4 feature schema in deployment bundle. "
        "The first 14 features and the final 4 TF-IDF features must "
        "appear in the frozen positional order."
    )

if len(FEATURES) != 18:
    raise RuntimeError(f"Expected 18 features, found {len(FEATURES)}")

if THRESHOLD != 0.85:
    raise RuntimeError(f"Unexpected production threshold: {THRESHOLD}. Expected 0.85.")

model_feature_count = model.get_feature_importance().shape[0]
if model_feature_count != 18:
    raise RuntimeError(f"CatBoost model expects {model_feature_count} features, not 18.")

# This is the critical positional check. build_v4_matrix() below creates:
#   14 V2 features
#   name-word TF-IDF cosine
#   address-word TF-IDF cosine
#   name-char TF-IDF cosine
#   address-char TF-IDF cosine
# Therefore the matrix order matches either accepted naming schema exactly.

print("=" * 70)
print("PRODUCTION BUNDLE LOADED")
print("=" * 70)
print("Model:", type(model).__name__)
print("Feature count:", len(FEATURES))
print("Threshold:", THRESHOLD)
print("Model feature count:", model_feature_count)
print("Bundle feature names (authoritative):")
for i, feature in enumerate(FEATURES, start=1):
    print(f"{i:2d}. {feature}")

print("✓ 18-feature positional recipe verified.")
print("✓ Bundle feature names accepted as authoritative.")
print("✓ No model retraining or feature renaming is being performed.")


PRODUCTION BUNDLE LOADED
Model: CatBoostClassifier
Feature count: 18
Threshold: 0.85
Model feature count: 18
Bundle feature names (authoritative):
 1. name_edit_similarity
 2. name_token_jaccard
 3. address_edit_similarity
 4. address_token_jaccard
 5. country_match
 6. name_exact_normalized
 7. name_exact_legal
 8. name_length_ratio
 9. address_exact_normalized
10. address_length_ratio
11. s1_name_missing
12. match_name_missing
13. s1_address_missing
14. match_address_missing
15. name_word_tfidf_cosine
16. address_word_tfidf_cosine
17. name_char_tfidf_cosine
18. address_char_tfidf_cosine
✓ 18-feature positional recipe verified.
✓ Bundle feature names accepted as authoritative.
✓ No model retraining or feature renaming is being performed.


In [4]:
# ============================================================
# 4. VERIFY FITTED TF-IDF RECIPES
# ============================================================

for label, vec in [
    ("Name word", name_vectorizer),
    ("Address word", address_vectorizer),
    ("Name char", char_name_vectorizer),
    ("Address char", char_address_vectorizer),
]:
    print(f"\n{label} TF-IDF:")
    print(" analyzer     :", vec.analyzer)
    print(" ngram_range  :", vec.ngram_range)
    print(" min_df       :", vec.min_df)
    print(" max_features :", vec.max_features)
    print(" vocabulary   :", len(vec.vocabulary_))

print("\n✓ Using fitted production vectorizers. No refitting will occur.")


Name word TF-IDF:
 analyzer     : word
 ngram_range  : (1, 2)
 min_df       : 2
 max_features : 100000
 vocabulary   : 48854

Address word TF-IDF:
 analyzer     : word
 ngram_range  : (1, 2)
 min_df       : 2
 max_features : 100000
 vocabulary   : 100000

Name char TF-IDF:
 analyzer     : char
 ngram_range  : (2, 5)
 min_df       : 2
 max_features : 150000
 vocabulary   : 150000

Address char TF-IDF:
 analyzer     : char
 ngram_range  : (2, 5)
 min_df       : 2
 max_features : 150000
 vocabulary   : 150000

✓ Using fitted production vectorizers. No refitting will occur.


## 5. Load the complete test entities lazily

Only the columns required by the matcher are retained. `match_id` can point to either Source 2 or Source 3.

In [10]:
# ============================================================
# 5. LAZY TEST ENTITY LOOKUPS
# ============================================================

test_s1 = pl.scan_csv(FILES["test1"], separator="\t")
test_s2 = pl.scan_csv(FILES["test2"], separator="\t")
test_s3 = pl.scan_csv(FILES["test3"], separator="\t")

s1_test_lookup = (
    test_s1
    .select(["entity_id", "business_name", "business_address", "country"])
    .rename({
        "entity_id": "source1_entity_id",
        "business_name": "s1_name",
        "business_address": "s1_address",
        "country": "s1_country",
    })
)

s2_test_lookup = (
    test_s2
    .select(["entity_id", "business_name", "business_address", "country"])
    .rename({
        "entity_id": "match_id",
        "business_name": "match_name",
        "business_address": "match_address",
        "country": "match_country",
    })
)

s3_test_lookup = (
    test_s3
    .select(["entity_id", "business_name", "business_address", "country"])
    .rename({
        "entity_id": "match_id",
        "business_name": "match_name",
        "business_address": "match_address",
        "country": "match_country",
    })
)

candidate_test_lookup = pl.concat([s2_test_lookup, s3_test_lookup])

total_test_s1 = test_s1.select(pl.len()).collect().item()
total_test_s2 = test_s2.select(pl.len()).collect().item()
total_test_s3 = test_s3.select(pl.len()).collect().item()

print("Test S1:", f"{total_test_s1:,}")
print("Test S2:", f"{total_test_s2:,}")
print("Test S3:", f"{total_test_s3:,}")
print("\n✓ Test lookup tables prepared lazily.")

Test S1: 1,732,544
Test S2: 4,887,273
Test S3: 5,082,316

✓ Test lookup tables prepared lazily.


In [11]:
# ============================================================
# 6. EXACT V4 NON-TFIDF FEATURE FUNCTIONS
# ============================================================

def normalize_text(value):
    if value is None:
        return ""
    value = str(value).lower()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def normalize_legal_name(value):
    value = normalize_text(value)
    legal_suffixes = [
        "private limited", "pvt ltd", "pvt limited", "private ltd",
        "limited", "ltd", "llc", "incorporated", "inc", "corp",
        "corporation", "company", "co",
    ]
    tokens = value.split()
    while tokens:
        removed = False
        for suffix in legal_suffixes:
            suffix_tokens = suffix.split()
            if len(tokens) >= len(suffix_tokens) and tokens[-len(suffix_tokens):] == suffix_tokens:
                tokens = tokens[:-len(suffix_tokens)]
                removed = True
                break
        if not removed:
            break
    return " ".join(tokens)


def safe_length_ratio(a, b):
    a = a or ""
    b = b or ""
    if not a or not b:
        return 0.0
    return min(len(a), len(b)) / max(len(a), len(b))


def exact_normalized(a, b):
    a = normalize_text(a)
    b = normalize_text(b)
    if not a or not b:
        return 0
    return int(a == b)


def exact_legal_name(a, b):
    a = normalize_legal_name(a)
    b = normalize_legal_name(b)
    if not a or not b:
        return 0
    return int(a == b)


def token_jaccard(a, b):
    if not a or not b:
        return 0.0
    a_tokens = set(a.split())
    b_tokens = set(b.split())
    if not a_tokens or not b_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def edit_similarity(a, b):
    if not a or not b:
        return 0.0
    return ratio(a, b) / 100.0


def add_v2_features(df):
    return df.with_columns([
        pl.struct(["s1_name", "match_name"]).map_elements(
            lambda x: edit_similarity(x["s1_name"], x["match_name"]),
            return_dtype=pl.Float64,
        ).alias("name_edit_similarity"),

        pl.struct(["s1_name", "match_name"]).map_elements(
            lambda x: token_jaccard(x["s1_name"], x["match_name"]),
            return_dtype=pl.Float64,
        ).alias("name_token_jaccard"),

        pl.struct(["s1_address", "match_address"]).map_elements(
            lambda x: edit_similarity(x["s1_address"], x["match_address"]),
            return_dtype=pl.Float64,
        ).alias("address_edit_similarity"),

        pl.struct(["s1_address", "match_address"]).map_elements(
            lambda x: token_jaccard(x["s1_address"], x["match_address"]),
            return_dtype=pl.Float64,
        ).alias("address_token_jaccard"),

        (pl.col("s1_country") == pl.col("match_country")).cast(pl.Int8).alias("country_match"),

        pl.struct(["s1_name", "match_name"]).map_elements(
            lambda x: exact_normalized(x["s1_name"], x["match_name"]),
            return_dtype=pl.Int8,
        ).alias("name_exact_normalized"),

        pl.struct(["s1_name", "match_name"]).map_elements(
            lambda x: exact_legal_name(x["s1_name"], x["match_name"]),
            return_dtype=pl.Int8,
        ).alias("name_exact_legal"),

        pl.struct(["s1_name", "match_name"]).map_elements(
            lambda x: safe_length_ratio(x["s1_name"], x["match_name"]),
            return_dtype=pl.Float64,
        ).alias("name_length_ratio"),

        pl.struct(["s1_address", "match_address"]).map_elements(
            lambda x: exact_normalized(x["s1_address"], x["match_address"]),
            return_dtype=pl.Int8,
        ).alias("address_exact_normalized"),

        pl.struct(["s1_address", "match_address"]).map_elements(
            lambda x: safe_length_ratio(x["s1_address"], x["match_address"]),
            return_dtype=pl.Float64,
        ).alias("address_length_ratio"),

        (pl.col("s1_name").fill_null("").str.strip_chars() == "").cast(pl.Int8).alias("s1_name_missing"),
        (pl.col("match_name").fill_null("").str.strip_chars() == "").cast(pl.Int8).alias("match_name_missing"),
        (pl.col("s1_address").fill_null("").str.strip_chars() == "").cast(pl.Int8).alias("s1_address_missing"),
        (pl.col("match_address").fill_null("").str.strip_chars() == "").cast(pl.Int8).alias("match_address_missing"),
    ])

print("✓ Exact V2 feature functions ready.")

✓ Exact V2 feature functions ready.


In [12]:
# ============================================================
# 7. TF-IDF COSINE HELPERS + EXACT V4 MATRIX
# ============================================================

def rowwise_cosine_similarity(A, B):
    # TfidfVectorizer uses L2 normalization by default.
    return np.asarray(A.multiply(B).sum(axis=1)).ravel().astype(np.float32)


def transform_pairwise_tfidf(vectorizer, left_text, right_text):
    left_matrix = vectorizer.transform(left_text)
    right_matrix = vectorizer.transform(right_text)
    result = rowwise_cosine_similarity(left_matrix, right_matrix)
    del left_matrix, right_matrix
    return result


def build_v4_matrix(enriched_df):
    v2 = add_v2_features(enriched_df)

    s1_names = v2["s1_name"].fill_null("").to_list()
    match_names = v2["match_name"].fill_null("").to_list()
    s1_addresses = v2["s1_address"].fill_null("").to_list()
    match_addresses = v2["match_address"].fill_null("").to_list()

    name_word = transform_pairwise_tfidf(name_vectorizer, s1_names, match_names)
    address_word = transform_pairwise_tfidf(address_vectorizer, s1_addresses, match_addresses)
    name_char = transform_pairwise_tfidf(char_name_vectorizer, s1_names, match_names)
    address_char = transform_pairwise_tfidf(char_address_vectorizer, s1_addresses, match_addresses)

    X_v2 = (
        v2
        .select([
            "name_edit_similarity",
            "name_token_jaccard",
            "address_edit_similarity",
            "address_token_jaccard",
            "country_match",
            "name_exact_normalized",
            "name_exact_legal",
            "name_length_ratio",
            "address_exact_normalized",
            "address_length_ratio",
            "s1_name_missing",
            "match_name_missing",
            "s1_address_missing",
            "match_address_missing",
        ])
        .to_numpy()
        .astype(np.float32)
    )

    X_v4 = np.column_stack([
        X_v2,
        name_word,
        address_word,
        name_char,
        address_char,
    ]).astype(np.float32, copy=False)

    if X_v4.shape != (enriched_df.height, 18):
        raise RuntimeError(f"Unexpected V4 matrix shape: {X_v4.shape}")
    if not np.isfinite(X_v4).all():
        raise RuntimeError("Non-finite values found in V4 inference matrix.")

    return X_v4

print("✓ V4 batch feature builder ready.")

✓ V4 batch feature builder ready.


## 8. Smoke test

Before touching all 64 partitions, run a bounded test through:

```text
candidate rows → enrichment → 18 features → CatBoost → threshold 0.85
```

If this fails, **stop here**.

In [17]:
# ============================================================
# 8. SMOKE TEST
# ============================================================

SMOKE_TEST_ROWS = 10_000
SMOKE_PATH = FINAL_CANDIDATE_DIR / "part_000.parquet"

smoke_candidates = (
    pl.scan_parquet(SMOKE_PATH)
    .head(SMOKE_TEST_ROWS)
    .collect(engine="streaming")
)

start = time.time()
smoke_enriched = (
    smoke_candidates.lazy()
    .join(s1_test_lookup, on="source1_entity_id", how="left")
    .join(candidate_test_lookup, on="match_id", how="left")
    .collect(engine="streaming")
)

print("Smoke candidates:", smoke_candidates.height)
print("Enriched rows:", smoke_enriched.height)
print("Enrichment time:", f"{time.time() - start:.2f}s")

null_check = smoke_enriched.select([
    pl.col("s1_name").is_null().sum().alias("missing_s1_name"),
    pl.col("match_name").is_null().sum().alias("missing_match_name"),
    pl.col("s1_address").is_null().sum().alias("missing_s1_address"),
    pl.col("match_address").is_null().sum().alias("missing_match_address"),
])

print("\nLookup null check:")
print(null_check)

# Only missing entity/name lookups are fatal.
# Missing addresses are legitimate and represented by V4 missing-value features.

missing_entities = (
    null_check.select([
        "missing_s1_name",
        "missing_match_name",
    ])
    .row(0)
)

if any(int(v) > 0 for v in missing_entities):
    raise RuntimeError(
        f"Entity lookup failure in smoke test: {missing_entities}"
    )

print(
    "✓ Entity lookup check passed. "
    f"Missing addresses: "
    f"S1={null_check['missing_s1_address'][0]}, "
    f"match={null_check['missing_match_address'][0]}"
)

X_smoke = build_v4_matrix(smoke_enriched)
smoke_prob = model.predict_proba(X_smoke)[:, 1]
smoke_pred = smoke_prob >= THRESHOLD

print("\n" + "=" * 65)
print("CATBOOST V4 SMOKE TEST")
print("=" * 65)
print("Rows:", len(smoke_prob))
print("Features:", X_smoke.shape[1])
print("Threshold:", THRESHOLD)
print("Predicted matches:", int(smoke_pred.sum()))
print("Positive rate:", f"{smoke_pred.mean():.6%}")
print("Probability min:", float(smoke_prob.min()))
print("Probability max:", float(smoke_prob.max()))
print("Probability mean:", float(smoke_prob.mean()))

if len(smoke_prob) != smoke_candidates.height or not np.isfinite(smoke_prob).all():
    raise RuntimeError("Smoke-test prediction integrity failure.")

print("\n✓ Smoke test passed.")

Smoke candidates: 10000
Enriched rows: 10000
Enrichment time: 2.97s

Lookup null check:
shape: (1, 4)
┌─────────────────┬────────────────────┬────────────────────┬───────────────────────┐
│ missing_s1_name ┆ missing_match_name ┆ missing_s1_address ┆ missing_match_address │
│ ---             ┆ ---                ┆ ---                ┆ ---                   │
│ u32             ┆ u32                ┆ u32                ┆ u32                   │
╞═════════════════╪════════════════════╪════════════════════╪═══════════════════════╡
│ 0               ┆ 0                  ┆ 0                  ┆ 0                     │
└─────────────────┴────────────────────┴────────────────────┴───────────────────────┘
✓ Entity lookup check passed. Missing addresses: S1=0, match=0

CATBOOST V4 SMOKE TEST
Rows: 10000
Features: 18
Threshold: 0.85
Predicted matches: 106
Positive rate: 1.060000%
Probability min: 1.1484706886736289e-08
Probability max: 0.999992797040299
Probability mean: 0.019989993743607302

✓ Smo

## 9. Full inference engine

For each existing candidate partition:

```text
partition
  ↓
100k bounded batches
  ↓
entity enrichment
  ↓
exact 18-feature V4 matrix
  ↓
CatBoost predict_proba
  ↓
P(match) >= 0.85
  ↓
positive pair Parquet files
```

Positive batches are written to disk rather than accumulated in RAM.

In [ ]:
# ============================================================
# 9. PARTITIONED CATBOOST INFERENCE
# ============================================================

import shutil
import gc
import time
# ============================================================
# USER CONTROLS
# ============================================================

# Partition range
# START = inclusive
# END   = exclusive
#
# Examples:
#   0, 1    -> partition 0
#   0, 10   -> partitions 0-9
#   10, 20  -> partitions 10-19
#   0, 64   -> all 64 partitions

PARTITION_START = 0
PARTITION_END   = 1

# Batch size
#
# Start with 250k.
# If stable, try 500k.
# If memory becomes a problem, reduce to 100k.

BATCH_SIZE = 200_000

# IMPORTANT:
# True  -> delete ALL previous inference outputs
# False -> preserve previous partition outputs
#
# Use True ONLY for a completely fresh inference run.
CLEAN_INFERENCE_OUTPUT = False

# ============================================================
# CONFIGURATION DISPLAY
# ============================================================

print("=" * 72)
print("CATBOOST INFERENCE CONFIGURATION")
print("=" * 72)
print(
    f"Partitions : "
    f"{PARTITION_START} → {PARTITION_END - 1}"
)

print(f"Batch size : {BATCH_SIZE:,}")

print(
    f"Clean old inference outputs: "
    f"{CLEAN_INFERENCE_OUTPUT}"
)

print("=" * 72)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

if CLEAN_INFERENCE_OUTPUT:

    if INFERENCE_DIR.exists():
        print("\nRemoving previous inference outputs...")
        shutil.rmtree(INFERENCE_DIR)

    print("✓ Old inference outputs removed.")
INFERENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)
print(
    "\nInference output directory:",
    INFERENCE_DIR.resolve()
)
# ============================================================
# GLOBAL TIMER
# ============================================================

global_start = time.time()
partition_summaries = []
# ============================================================
# PARTITION LOOP
# ===========================================================
for part_idx in range(
    PARTITION_START,
    PARTITION_END
):
    partition_path = (
        FINAL_CANDIDATE_DIR /
        f"part_{part_idx:03d}.parquet"
    )
    partition_out_dir = (
        INFERENCE_DIR /
        f"part_{part_idx:03d}"
    )
    partition_out_dir.mkdir(
        parents=True,
        exist_ok=True
    )
    # -------------------------------------------------------
    # VERIFY PARTITION EXISTS
    # --------------------------------------------------------

    if not partition_path.exists():
        raise FileNotFoundError(
            f"\nCandidate partition not found:\n"
            f"{partition_path}"
        )
    # --------------------------------------------------------
    # COUNT PARTITION ROWS
    # --------------------------------------------------------

    partition_rows = (
        pl.scan_parquet(partition_path)
        .select(pl.len())
        .collect()
        .item()
    )
    print()
    print("=" * 72)
    print(
        f"PARTITION "
        f"{part_idx + 1}/{N_PARTITIONS} | "
        f"{partition_rows:,} candidate rows"
    )
    print("=" * 72)
    # --------------------------------------------------------
    # PARTITION STATE
    # --------------------------------------------------------
    part_start = time.time()
    part_candidates = 0
    part_positives = 0
    batch_count = 0
    # --------------------------------------------------------
    # BATCH LOOP
    # --------------------------------------------------------

    for offset in range(
        0,
        partition_rows,
        BATCH_SIZE
    ):
        batch_end = min(
            offset + BATCH_SIZE,
            partition_rows
        )
        # ====================================================
        # LOAD CANDIDATE BATCH
        # ====================================================

        batch = (
            pl.scan_parquet(partition_path)
            .slice(
                offset,
                batch_end - offset
            )
            .collect(
                engine="streaming"
            )
        )
        # ====================================================
        # ENTITY ENRICHMENT
        # ====================================================
        enriched = (
            batch.lazy()
            .join(
                s1_test_lookup,
                on="source1_entity_id",
                how="left"
            )
            .join(
                candidate_test_lookup,
                on="match_id",
                how="left"
            )
            .collect(
                engine="streaming"
            )
        )
        # ==================================================
        # ENTITY LOOKUP INTEGRITY CHECK
        # ====================================================
        #
        # Missing NAME = actual lookup failure.
        #
        # Missing ADDRESS = legitimate source-data condition.
        # Those are handled by the missing-address features.

        missing_entity_lookup = (
            enriched
            .select([
                pl.col("s1_name")
                .is_null()
                .sum()
                .alias("missing_s1_name"),

                pl.col("match_name")
                .is_null()
                .sum()
                .alias("missing_match_name"),
            ])

            .row(0)
        )
        if any(
            int(v) > 0
            for v in missing_entity_lookup
        ):
            raise RuntimeError(
                f"Entity lookup failure in "
                f"partition {part_idx}, "
                f"batch {offset}: "
                f"{missing_entity_lookup}"
            )
        # ====================================================
        # ADDRESS NULL CHECK
        # ====================================================
        missing_address_counts = (
            enriched

            .select([
                pl.col("s1_address")
                .is_null()
                .sum()
                .alias("missing_s1_address"),

                pl.col("match_address")
                .is_null()
                .sum()
                .alias("missing_match_address"),
            ])

            .row(0)
        )
        if any(
            int(v) > 0
            for v in missing_address_counts
        ):
            print(
                f"  Missing addresses: "
                f"S1={missing_address_counts[0]}, "
                f"match={missing_address_counts[1]}"
            )
        # ====================================================
        # BUILD V4 FEATURE MATRIX
        # ====================================================
        X_batch = build_v4_matrix(
            enriched
        )
        # ====================================================
        # CATBOOST PREDICTION
        # ====================================================
        probabilities = (
            model
            .predict_proba(X_batch)[:, 1]
        )
        # ====================================================
        # APPLY FINAL THRESHOLD
        # ====================================================
        positive_mask = (
            probabilities >= THRESHOLD
        )
        positive_count = int(
            positive_mask.sum()
        )
        # ====================================================
        # WRITE POSITIVE MATCHES TO DISK
        # ====================================================
        if positive_count > 0:
            positive = (
                enriched
                .select([
                    "source1_entity_id",
                    "match_id"
                ])
                .filter(
                    pl.Series(
                        "positive",
                        positive_mask
                    )
                )
                .with_columns(
                    pl.Series(
                        "probability",
                        probabilities[
                            positive_mask
                        ]
                    )
                )
            )
            batch_out = (
                partition_out_dir /
                f"batch_{batch_count:04d}.parquet"
            )
            positive.write_parquet(
                batch_out,
                compression="zstd"
            )
            part_positives += (
                positive_count
            )
        # ====================================================
        # UPDATE COUNTERS
        # ====================================================
        part_candidates += (
            batch.height
        )
        batch_count += 1
        # ====================================================
        # PROGRESS
        # ====================================================

        elapsed = (
            time.time() -
            part_start
        )


        print(
            f"  batch {batch_count:4d} | "
            f"{batch_end:,}/{partition_rows:,} | "
            f"positives={positive_count:,} | "
            f"part positives={part_positives:,} | "
            f"{elapsed / 60:.1f} min"
        )
        # ====================================================
        # MEMORY CLEANUP
        # ====================================================
        del (
            batch,
            enriched,
            X_batch,
            probabilities,
            positive_mask
        )
        if positive_count > 0:
            del positive
        gc.collect()
    # ========================================================
    # PARTITION COMPLETE
    # ========================================================

    part_elapsed = (
        time.time() -
        part_start
    )

    partition_summaries.append({
        "partition": part_idx,
        "candidate_rows":
            part_candidates,
        "positive_rows":
            part_positives,
        "batches":
            batch_count,
        "minutes":
            part_elapsed / 60,
    })
    print()
    print(
        f"✓ Partition {part_idx:03d} COMPLETE | "
        f"candidates={part_candidates:,} | "
        f"positives={part_positives:,} | "
        f"time={part_elapsed / 60:.2f} min"
    )
# ============================================================
# COMPLETE
# ============================================================
total_elapsed = (
    time.time() -
    global_start
)
print()
print("=" * 72)
print("SELECTED CATBOOST INFERENCE COMPLETE")
print("=" * 72)
print(
    f"Partitions processed: "
    f"{PARTITION_START} → "
    f"{PARTITION_END - 1}"
)
print(
    f"Batch size: "
    f"{BATCH_SIZE:,}"
)
print(
    f"Total time: "
    f"{total_elapsed / 3600:.2f} hours"
)
print("=" * 72)

CATBOOST INFERENCE CONFIGURATION
Partitions : 0 → 0
Batch size : 200,000
Clean old inference outputs: False

Inference output directory: F:\coding\amazonML2026\submissions\artifacts\inference_matches

PARTITION 1/64 | 25,175,214 candidate rows
  Missing addresses: S1=0, match=300
  batch    1 | 200,000/25,175,214 | positives=7,461 | part positives=7,461 | 1.5 min
  Missing addresses: S1=0, match=329


In [ ]:
# ============================================================
# 10. INFERENCE VOLUME SANITY CHECK
# ============================================================

summary_df = pl.DataFrame(partition_summaries)
total_candidates = summary_df["candidate_rows"].sum()
total_positive_rows = summary_df["positive_rows"].sum()

print(summary_df)
print("\n" + "=" * 70)
print("INFERENCE VOLUME")
print("=" * 70)
print("Candidate rows processed :", f"{total_candidates:,}")
print("Positive rows written    :", f"{total_positive_rows:,}")
print("Overall positive rate    :", f"{total_positive_rows / total_candidates:.6%}")

if total_positive_rows == 0:
    raise RuntimeError("Zero predictions passed threshold. Do not generate a submission.")

print("\n✓ Inference produced predictions.")

## 11. Aggregate predictions by S1

Candidate partitions are partitioned by `source1_entity_id`, so each S1 belongs to exactly one partition. We aggregate one partition at a time and write compact partition-level results.

In [ ]:
# ============================================================
# 11. PARTITION-LEVEL MATCH AGGREGATION
# ============================================================

AGG_DIR = ARTIFACT_DIR / "inference_matches_aggregated"
if AGG_DIR.exists():
    shutil.rmtree(AGG_DIR)
AGG_DIR.mkdir(parents=True, exist_ok=True)

for part_idx in range(N_PARTITIONS):
    part_dir = INFERENCE_DIR / f"part_{part_idx:03d}"
    batch_files = sorted(part_dir.glob("batch_*.parquet"))
    out_path = AGG_DIR / f"part_{part_idx:03d}.parquet"

    if not batch_files:
        pl.DataFrame({
            "source1_entity_id": pl.Series([], dtype=pl.String),
            "matched_entity_ids": pl.Series([], dtype=pl.String),
        }).write_parquet(out_path)
        continue

    positives = pl.scan_parquet([str(p) for p in batch_files])

    aggregated = (
        positives
        .select(["source1_entity_id", "match_id"])
        .unique()
        .group_by("source1_entity_id")
        .agg(
            pl.col("match_id").sort().str.join(",").alias("matched_entity_ids")
        )
        .collect(engine="streaming")
    )

    aggregated.write_parquet(out_path, compression="zstd")
    print(f"[{part_idx + 1:02d}/{N_PARTITIONS}] S1 with predictions={aggregated.height:,}")

print("✓ Partition-level aggregation complete.")

## 12. Build the final submission table

**Critical rule:** start from the complete test S1 population.

This guarantees:

- every S1 appears
- S1s with zero predicted matches are retained
- every S1 appears exactly once
- no S1 disappears merely because the model predicted nothing

In [ ]:
# ============================================================
# 12. COMPLETE-S1 LEFT JOIN + FINAL OUTPUT
# ============================================================

complete_s1 = (
    s1_test_lookup
    .select("source1_entity_id")
    .collect(engine="streaming")
)

aggregated_files = sorted(AGG_DIR.glob("part_*.parquet"))
if len(aggregated_files) != N_PARTITIONS:
    raise RuntimeError(
        f"Expected {N_PARTITIONS} aggregate partitions, found {len(aggregated_files)}"
    )

all_predicted = pl.scan_parquet([str(p) for p in aggregated_files])

matching_results = (
    complete_s1.lazy()
    .join(all_predicted, on="source1_entity_id", how="left")
    .with_columns(pl.col("matched_entity_ids").fill_null(""))
    .select(["source1_entity_id", "matched_entity_ids"])
    .collect(engine="streaming")
)

expected_rows = total_test_s1
actual_rows = matching_results.height
unique_s1 = matching_results["source1_entity_id"].n_unique()

if actual_rows != expected_rows:
    raise RuntimeError(f"Output row count mismatch: expected {expected_rows:,}, got {actual_rows:,}")
if unique_s1 != expected_rows:
    raise RuntimeError(f"Expected {expected_rows:,} unique S1 IDs, got {unique_s1:,}")

matched_s1 = matching_results.filter(pl.col("matched_entity_ids") != "").height
unmatched_s1 = actual_rows - matched_s1

print("=" * 70)
print("FINAL MATCHING RESULTS CHECK")
print("=" * 70)
print("Expected S1 entities :", f"{expected_rows:,}")
print("Output rows          :", f"{actual_rows:,}")
print("Unique S1 IDs        :", f"{unique_s1:,}")
print("S1 with matches      :", f"{matched_s1:,}")
print("S1 without matches   :", f"{unmatched_s1:,}")
print("\n✓ Every Source 1 entity appears exactly once.")
print("✓ S1 entities without predictions are retained.")
print("✓ No duplicate S1 rows.")

In [ ]:
# ============================================================
# 13. FINAL SUBMISSION FILE
# ============================================================

MATCHING_RESULTS_PATH = OUTPUT_DIR / "matching_results.tsv"

matching_results.write_csv(MATCHING_RESULTS_PATH, separator="\t")

print("Saved:", MATCHING_RESULTS_PATH.resolve())
print("Exists:", MATCHING_RESULTS_PATH.exists())
print("Size:", f"{MATCHING_RESULTS_PATH.stat().st_size / 1024**2:.2f} MB")
print("\nSample:")
print(matching_results.head(10))

In [ ]:
# ============================================================
# 14. FINAL SUBMISSION SANITY CHECK
# ============================================================

reloaded = pl.read_csv(MATCHING_RESULTS_PATH, separator="\t")

expected_columns = ["source1_entity_id", "matched_entity_ids"]
if reloaded.columns != expected_columns:
    raise RuntimeError(f"Unexpected submission columns: {reloaded.columns}")
if reloaded.height != total_test_s1:
    raise RuntimeError(f"Submission row count mismatch: {reloaded.height:,} vs {total_test_s1:,}")
if reloaded["source1_entity_id"].n_unique() != total_test_s1:
    raise RuntimeError("Submission does not contain exactly one row per S1 entity.")
if reloaded["source1_entity_id"].null_count() != 0:
    raise RuntimeError("Null Source 1 IDs found.")
if reloaded["matched_entity_ids"].null_count() != 0:
    raise RuntimeError("Null matched_entity_ids found.")

print("=" * 70)
print("FINAL SUBMISSION SANITY CHECK PASSED")
print("=" * 70)
print("File:", MATCHING_RESULTS_PATH.resolve())
print("Rows:", f"{reloaded.height:,}")
print("Columns:", reloaded.columns)
print("Size:", f"{MATCHING_RESULTS_PATH.stat().st_size / 1024**2:.2f} MB")
print("\n✓ READY FOR MANUAL SUBMISSION")

# Final handoff

The only leaderboard submission file produced by this notebook is:

```text
output/matching_results.tsv
```

### Frozen production recipe

```text
Existing final candidate partitions
        ↓
bounded inference batches
        ↓
exact V4 18-feature positional recipe
        ↓
fitted production TF-IDF vectorizers
        ↓
CatBoost V4 Depth-10
        ↓
P(match) >= 0.85
        ↓
deduplicate candidate pairs
        ↓
aggregate matches by S1
        ↓
LEFT JOIN complete test S1 population
        ↓
matching_results.tsv
```

**Do not regenerate candidate partitions from this notebook.**

**Do not refit TF-IDF from test data.**

**Do not retrain CatBoost locally.**

**Do not change the threshold during final inference.**